#Transform Circuits Data
1. Read bronze circuits table
2. Keep only the columns required for analytics (Drop url column)
3. Standardize column names using snake_case (circuitId → circuit_id, circuitName → circuit_name)
4. Rename columns to make them more meaningful (lat → latitude, long → longitude)
5. Filter records where circuit_id  is NULL
6. Remove duplicate records
7. Transform column values of circuit_name and locality to Title Case
8. Write the transformed data to silver circuits table

###Below changes are required to implement Incremental Load Processing

1. Accept batch_id as a parameter to the notebook

2. Process data for only the batch_id being passed in (i.e., filter reading from bronze using the batch_id)
3. Add created_timestamp, updated_timestamp and batch_id to the silver table.
4. Merge the processed data to the silver table
    - created_timestamp should only be populated at the time of inserting/ creating the record.It should not be updated during the merge update.
    - Ensure that we are not overwriting the data in silver table by older bronze data (re-run scenario)

In [0]:
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment_config

In [0]:
%run ../00-common/03.silver-helpers

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.circuits"
silver_table = f"{catalog_name}.{silver_schema}.circuits"


Step 1-Read bronze circuits table

In [0]:
circuits_df = (
    spark.table(bronze_table).filter((F.col("batch_id")== v_batch_id ))
    )
#                            (OR)
# circuits_df = spark.read.option("versionAsOf",0).table(bronze_table)

In [0]:
display(circuits_df)

Step 2- Keep only the columns required for analytics (Drop url column)

In [0]:

circuits_selected_df = circuits_df.select(
    F.col("circuitId"),
    F.col("circuitName"),
    F.col("lat"),
    F.col("long"),
    F.col("locality"),
    F.col("country"),
    F.col("ingestion_timestamp"),
    F.col("source_file"),
    F.col("batch_id")
)


In [0]:
display(circuits_selected_df)

Step 3 & 4:
- Standardize column names using snake_case (circuitId → circuit_id, circuitName → circuit_name)
- Rename columns to make them more meaningful (lat → latitude, long → longitude)

In [0]:
"""circuits_renamed_df = (

        circuits_selected_df
            .withColumnRenamed("circuitId","circuit_id")
            .withColumnRenamed("circuitName","circuit_name")
            .withColumnRenamed("lat","latitude")
            .withColumnRenamed("long","longitude")  
                                                    )

                      OR                             """
circuits_renamed_df = (circuits_selected_df
                       .withColumnsRenamed({"circuitId":"circuit_id","circuitName":"circuit_name","lat":"latitude","long":"longitude"}))





Step 5- Filter records where circuit_id is NULL (business Key validation)

In [0]:
"""circuits_valid_df = circuits_renamed_df.filter(
    "circuit_id IS NOT NULL"
)
circuits_valid_df.display()  """

circuits_valid_df = circuits_renamed_df.filter(
    F.col("circuit_id").isNotNull()
)
circuits_valid_df.display()

Step 6- Remove duplicate records

In [0]:
"""circuits_distinct_df = circuits_valid_df.distinct()
                        OR  """
circuits_distinct_df = circuits_valid_df.dropDuplicates(["circuit_id"])
circuits_distinct_df.display()

Step 7- Transform column values of circuit_name and locality to Title Case

In [0]:
circuits_final_df = (circuits_distinct_df
                            .withColumn("circuit_name",F.initcap(F.col("circuit_name")))
                            .withColumn("locality",F.initcap(F.col("locality")))

)
circuits_final_df.display()



Step 8- Write the transformed data to silver circuits table

In [0]:
# circuits_final_df = (
#     circuits_final_df
#         .withColumn("created_timestamp", F.current_timestamp())
#         .withColumn("updated_timestamp", F.current_timestamp())
# )

In [0]:
# if not spark.catalog.tableExists(silver_table):
#     (
#         circuits_final_df
#             .write
#             .format("delta")
#             .mode("overwrite")
#             .saveAsTable(silver_table)
#     )

# else:
#     from delta.tables import DeltaTable

#     delta_table = DeltaTable.forName(spark, silver_table)
#     (
#         delta_table.alias("t")
#         .merge (
#             circuits_final_df.alias("s"),
#             "t.circuit_id = s.circuit_id"
#         )
#         .whenMatchedUpdate(
#             condition="s.batch_id >= t.batch_id",
#             set = {
#                     "circuit_name": "s.circuit_name",
#                     "latitude": "s.latitude",
#                     "longitude": "s.longitude",
#                     "locality": "s.locality",
#                     "country": "s.country",
#                     "ingestion_timestamp": "s.ingestion_timestamp",
#                     "source_file": "s.source_file",
#                     "batch_id": "s.batch_id",
#                     "updated_timestamp": "s.updated_timestamp"              
#             }
#         )
#         .whenNotMatchedInsertAll()
#         .execute()
#     )

In [0]:
write_to_silver(
    input_df=circuits_final_df,
    target_table=silver_table,
    merge_condition="t.circuit_id = s.circuit_id",
    columns_to_update=[
        "circuit_name",
        "latitude",
        "longitude",
        "locality",
        "country",
        "ingestion_timestamp",
        "source_file",
        "batch_id"        
    ]
)

In [0]:
display(spark.table(silver_table))